In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
from scipy.signal import convolve2d


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
s = np.load('psf.npz')
psf = s['psf']


def calc_fluxes_(file, dlat=1, mu_thr=0.1, binning=4):
    with fits.open(file) as hdul:
        header = hdul[1].header.copy()
        data = hdul[1].data.copy()

    data = rebin(data, binning, update_header=header)
    #data = convolve2d(data, psf, mode='same', boundary='symm')

    view = View.from_header(header)

    transform = view.to_carrington(correct_mu=True, mu_thr=mu_thr) + ToSpherical()
    grid, mu = transform(view.grid())

    Br = data / mu
    lat_map = grid[0] // dlat * dlat

    lat = np.arange(-90,90,dlat)
    weight = np.zeros_like(lat).astype(np.float32)
    flux_density = np.zeros_like(lat).astype(np.float32)

    for i, lat_ in enumerate(lat):
        t = (lat_map == lat_)
        nt = np.sum(t)

        if nt > 0:
            Br_ = Br[t]
            W_ = mu[t] ** 4 / mu[t]

            weight[i] = np.nansum(W_)
            flux_density[i] = np.nansum(W_ * Br_) / weight[i]

    return lat, weight, flux_density


def calc_fluxes(files, **kwargs):
    x, mean, w_sum, w_sum2, S = 0., 0., 0. ,0., 0.

    for file in files:
        print(file)
        x, w, y = calc_fluxes_(file, **kwargs)

        w_sum += w
        w_sum2 += w ** 2
        mean_old = mean + 0.
        mean += np.nan_to_num((w / w_sum) * (y - mean))
        S += np.nan_to_num(w * (y - mean_old) * (y - mean))

    S *= w_sum2 / w_sum ** 3
    return x, mean, S


In [11]:
files = sorted(glob.glob('/home/ulyanov/data/sdo/hmi/2024-11/*'))
len(files)

356

In [12]:
with fits.open(files[0]) as hdul:
    header = hdul[1].header.copy()

header['CRLN_OBS']

226.48877

In [14]:
with fits.open(files[-1]) as hdul:
    header = hdul[1].header.copy()

header['CRLN_OBS']

192.076981

In [15]:
x, mean, S = calc_fluxes(files)

/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241027_000000_TAI.3.magnetogram.fits


/tmp/ipykernel_142549/210174047.py:34: RuntimeWarning: invalid value encountered in scalar divide
  flux_density[i] = np.nansum(W_ * Br_) / weight[i]
/tmp/ipykernel_142549/210174047.py:49: RuntimeWarning: invalid value encountered in divide
  mean += np.nan_to_num((w / w_sum) * (y - mean))


/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241027_020000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241027_040000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241027_060000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241027_080000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241027_100000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241027_120000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241027_140000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241027_160000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241027_180000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241027_200000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241027_220000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/2024-11/hmi.M_720s.20241028_000000_TAI

/tmp/ipykernel_142549/210174047.py:52: RuntimeWarning: invalid value encountered in divide
  S *= w_sum2 / w_sum ** 3


In [16]:
np.savez('fluxes_202411hmi.npz', lat=x, flux_density=mean, variance=S)